# Standard Output Tracers

The `stdout.py` module defines synchronous tracers that format LangChain run events as readable text.

`FunctionCallbackHandler` sends each formatted message to a user-supplied single-string callback. `ConsoleCallbackHandler` specializes it by using Python's `print` function.

## Constants

1. `MILLISECONDS_IN_SECOND`: Stores the number of milliseconds in one second.
   * **Definition:**
     ```python
     MILLISECONDS_IN_SECOND = 1000
     ```

### Functions

1. `try_json_stringify`: Converts an object into an indented JSON string.

   Unicode characters are preserved. When JSON serialization raises an exception, the supplied fallback string is returned.

   * **Syntax:**
     ```python
     try_json_stringify(
         obj: Any, # Object to serialize
         fallback: str # Value returned when serialization fails
     ) -> str
     ```

2. `elapsed`: Returns the elapsed duration of a run-like object.

   The object must provide `start_time` and `end_time` attributes. Durations below one second are formatted as whole milliseconds; longer durations are formatted as seconds with two decimal places.

   * **Syntax:**
     ```python
     elapsed(
         run: Any # Object containing start_time and end_time
     ) -> str
     ```

# FunctionCallbackHandler

`FunctionCallbackHandler` is a synchronous tracer that formats run lifecycle activity and passes each generated string to a callback function.

Chain, language-model, and tool messages include coloured lifecycle labels and bolded descriptions. Nested runs are represented through parent-to-child breadcrumbs.

## Bases

- `BaseTracer`

## Attributes

1. `name`: Stores the callback handler's identifying name.
   * **Type:**
     ```python
     name: str = "function_callback_handler"
     ```

2. `function_callback`: Stores the single-string callback that receives formatted tracer messages.
   * **Type:**
     ```python
     function_callback: Callable[
         [str],
         None
     ]
     ```

### Methods

1. `__init__`: Creates a function-based callback handler.
   * **Syntax:**
     ```python
     __init__(
         self,
         function: Callable[
             [str],
             None
         ], # Callback receiving each formatted message
         **kwargs: Any # Additional BaseTracer arguments
     ) -> None
     ```

2. `_persist_run`: Implements the tracer persistence contract without performing an action.

   This tracer reports lifecycle events immediately through `function_callback` instead of storing completed root runs.

   * **Syntax:**
     ```python
     _persist_run(
         self,
         run: Run # Completed root run
     ) -> None
     ```

3. `get_parents`: Returns the available ancestors of a run.

   Parent identifiers are resolved through `run_map`, beginning with the immediate parent. Traversal stops when a run has no parent or a referenced parent is unavailable.

   * **Syntax:**
     ```python
     get_parents(
         self,
         run: Run # Run whose ancestors are requested
     ) -> list[Run]
     ```

4. `get_breadcrumbs`: Returns a root-to-current-run breadcrumb string.

   Each entry uses the format `<run_type>:<run_name>` and entries are separated by `" > "`.

   * **Syntax:**
     ```python
     get_breadcrumbs(
         self,
         run: Run # Run for which breadcrumbs are generated
     ) -> str
     ```

5. `_on_chain_start`: Sends a formatted chain-start message.

   The message contains a green lifecycle label, breadcrumbs, the capitalized run type, and JSON-formatted inputs.

   * **Syntax:**
     ```python
     _on_chain_start(
         self,
         run: Run # Chain run that has started
     ) -> None
     ```

6. `_on_chain_end`: Sends a formatted chain-completion message.

   The message contains a blue lifecycle label, breadcrumbs, elapsed time, the capitalized run type, and JSON-formatted outputs.

   * **Syntax:**
     ```python
     _on_chain_end(
         self,
         run: Run # Successfully completed chain run
     ) -> None
     ```

7. `_on_chain_error`: Sends a formatted chain-error message.

   The message contains a red lifecycle label, breadcrumbs, elapsed time, the capitalized run type, and the JSON-formatted error or fallback error label.

   * **Syntax:**
     ```python
     _on_chain_error(
         self,
         run: Run # Errored chain run
     ) -> None
     ```

8. `_on_llm_start`: Sends a formatted language-model start message.

   When the run input contains a `"prompts"` field, surrounding whitespace is removed from each prompt before serialization.

   * **Syntax:**
     ```python
     _on_llm_start(
         self,
         run: Run # Language-model run that has started
     ) -> None
     ```

9. `_on_llm_end`: Sends a formatted language-model completion message.

   The message contains a blue lifecycle label, breadcrumbs, elapsed time, and JSON-formatted model output.

   * **Syntax:**
     ```python
     _on_llm_end(
         self,
         run: Run # Successfully completed language-model run
     ) -> None
     ```

10. `_on_llm_error`: Sends a formatted language-model error message.

    The message contains a red lifecycle label, breadcrumbs, elapsed time, and the JSON-formatted error or fallback error label.

    * **Syntax:**
      ```python
      _on_llm_error(
          self,
          run: Run # Errored language-model run
      ) -> None
      ```

11. `_on_tool_start`: Sends a formatted tool-start message.

    The message contains a green lifecycle label, breadcrumbs, and the stripped value stored under `run.inputs["input"]`.

    * **Syntax:**
      ```python
      _on_tool_start(
          self,
          run: Run # Tool run that has started
      ) -> None
      ```

12. `_on_tool_end`: Sends a formatted tool-completion message when the run has output data.

    The message contains a blue lifecycle label, breadcrumbs, elapsed time, and the stripped string value stored under `run.outputs["output"]`. No message is sent when `run.outputs` is empty or false.

    * **Syntax:**
      ```python
      _on_tool_end(
          self,
          run: Run # Successfully completed tool run
      ) -> None
      ```

13. `_on_tool_error`: Sends a formatted tool-error message.

    The message contains a red lifecycle label, breadcrumbs, elapsed time, and the run's error value.

    * **Syntax:**
      ```python
      _on_tool_error(
          self,
          run: Run # Errored tool run
      ) -> None
      ```

# ConsoleCallbackHandler

`ConsoleCallbackHandler` is a `FunctionCallbackHandler` that sends formatted tracer messages to standard output through Python's `print` function.

## Bases

- `FunctionCallbackHandler`

## Attributes

1. `name`: Stores the callback handler's identifying name.
   * **Type:**
     ```python
     name: str = "console_callback_handler"
     ```

### Methods

1. `__init__`: Creates a console callback handler using `print` as its message callback.
   * **Syntax:**
     ```python
     __init__(
         self,
         **kwargs: Any # Additional FunctionCallbackHandler arguments
     ) -> None
     ```